# Figure 6 · Young reference model calibration

Held-out calibration of the ico4 linear MD reference adjusted for sex and local FreeSurfer thickness. This notebook uses completed leave-one-out results from 37 cognitively normal participants aged under 50.

## 1. Data sources

In [ ]:
from pathlib import Path
from statistics import NormalDist
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display

RUN_DIR = Path("/mnt/newStor/paros/paros_WORK/xintong/code/cortical_trajectory_analysis/analysis_results/figure6_new/outputs/full")
sources = {
    "LOO z-scores": RUN_DIR / "young_reference_loo_z_young.npy",
    "LOO burden": RUN_DIR / "young_reference_loo_depth_band_burden.csv",
}
display(pd.DataFrame({"Source": sources.keys(), "Path": map(str, sources.values())}))
z = np.load(sources["LOO z-scores"], mmap_mode="r", allow_pickle=False)
burden = pd.read_csv(sources["LOO burden"])

## 2. Metric verification

The full position axis is node-major with 21 depths per node. Per-participant valid and signed-tail counts are verified against the exported table before plotting. Observed coverage is averaged across participants. Normal-theory nominal coverage serves as a reference for the residual-SD scores.

In [ ]:
bands = {"Outer": slice(0, 7), "Middle": slice(7, 14), "Inner": slice(14, 21)}
if z.shape != (len(burden), 4687 * 21):
    raise ValueError(f"Expected full ico4 LOO results, got {z.shape}")

profiles = z.reshape(len(burden), 4687, 21)
normal = NormalDist()
nominal_levels = [0.50, 0.80, 0.90, 2 * normal.cdf(2) - 1]
bins = np.linspace(-5, 5, 101)
coverage_rows, tail_rows, histogram_rows = [], [], []
for band, depth_slice in {**bands, "All": slice(None)}.items():
    scores = profiles[:, :, depth_slice].reshape(len(burden), -1)
    finite = np.isfinite(scores)
    valid = finite.sum(axis=1)
    if (valid == 0).any():
        raise ValueError(f"Missing participant scores in {band}")
    positive = ((scores > 2) & finite).sum(axis=1)
    negative = ((scores < -2) & finite).sum(axis=1)
    if band != "All":
        prefix = band.lower()
        for suffix, actual in {
            "n_valid_positions": valid, "n_z_young_gt_2": positive,
            "n_z_young_lt_minus_2": negative,
        }.items():
            np.testing.assert_array_equal(actual, burden[f"{prefix}_{suffix}"])
    for row, (values, mask, n) in enumerate(zip(scores, finite, valid)):
        v = values[mask]
        for nominal in nominal_levels:
            cutoff = normal.inv_cdf((1 + nominal) / 2)
            observed = (np.abs(v) <= cutoff).mean()
            coverage_rows.append((band, row + 1, nominal, observed, 100 * (observed - nominal)))
        tail_rows.append((band, row + 1, n / scores.shape[1],
                          positive[row] / n, negative[row] / n,
                          1 - (positive[row] + negative[row]) / n))
        histogram_rows.append(pd.DataFrame({
            "Band": band, "LOO row": row + 1, "Left edge": bins[:-1],
            "Density": np.histogram(v, bins)[0] / (n * np.diff(bins)),
        }))
coverage = pd.DataFrame(coverage_rows, columns=["Band", "LOO row", "Nominal", "Observed", "Deviation (pp)"])
tails = pd.DataFrame(tail_rows, columns=["Band", "LOO row", "Valid fraction", "Positive tail", "Negative tail", "Within ±2"])
density = pd.concat(histogram_rows, ignore_index=True).groupby(["Band", "Left edge"], sort=False, as_index=False)["Density"].mean()
coverage_plot = coverage.groupby(["Band", "Nominal"], sort=False, as_index=False)[["Observed", "Deviation (pp)"]].mean()
metrics = ["Valid fraction", "Positive tail", "Negative tail", "Within ±2"]
summary = tails.groupby("Band", sort=False)[metrics].quantile([.25, .5, .75]).mul(100)
display(summary.round(2))

## 3. Plotting

### A. Coverage deviation and signed z-score distribution

In [ ]:
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10,
                     "legend.fontsize": 9, "savefig.facecolor": "white", "pdf.fonttype": 42})
colors = dict(zip([*bands, "All"], ["#235784", "#5085a5", "#8198aa", "#30363d"]))
fig_calibration, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(9, 3.4), constrained_layout=True)
for band, color in colors.items():
    c = coverage_plot.loc[coverage_plot["Band"] == band]
    ax_a.plot(100 * c["Nominal"], c["Deviation (pp)"], "o-", color=color, label=band)
    d = density.loc[density["Band"] == band]
    ax_b.stairs(d["Density"], bins, color=color, label=band)
ax_a.axhline(0, color="0.6", linestyle="--", linewidth=1)
ax_a.set(xlabel="Nominal coverage", ylabel="Observed − nominal (pp)")
ax_a.xaxis.set_major_formatter(PercentFormatter())
ax_a.set_title("A   Interval coverage", loc="left", fontweight="bold")
x = np.linspace(-5, 5, 400)
ax_b.plot(x, np.exp(-x*x/2) / np.sqrt(2*np.pi), "--", color="0.5", label="Standard normal")
for cutoff in (-2, 2):
    ax_b.axvline(cutoff, color="0.7", linewidth=0.8)
ax_b.set(xlabel="Held-out z-score", ylabel="Density")
ax_b.set_title("B   Signed z-score distribution", loc="left", fontweight="bold")
ax_b.legend(frameon=False)
plt.show()

### B. Depth-band tail rates and valid scoring coverage

In [ ]:
fig_tails, axes = plt.subplots(1, 2, figsize=(8, 3.3), constrained_layout=True)
for i, band in enumerate(bands):
    rows = tails.loc[tails["Band"] == band]
    for offset, metric, color in [(-.13, "Positive tail", "#235784"), (.13, "Negative tail", "#ac674d")]:
        values = 100 * rows[metric]
        axes[0].scatter(np.full(len(values), i + offset), values, s=12, alpha=.45,
                        color=color, label=metric if i == 0 else None)
        axes[0].plot(i + offset, values.median(), marker="_", color=color, markersize=12, markeredgewidth=2)
    values = 100 * rows["Valid fraction"]
    axes[1].scatter(np.full(len(values), i), values, s=14, color=colors[band], alpha=.55)
axes[0].axhline(100 * (1 - normal.cdf(2)), linestyle="--", color="0.55", linewidth=1, label="Standard normal")
axes[0].legend(frameon=False)
for ax, label in zip(axes, ["Tail rate", "Valid scoring coverage"]):
    ax.set_xticks(range(len(bands)), list(bands))
    ax.set_ylabel(label)
    ax.yaxis.set_major_formatter(PercentFormatter())
plt.show()

# Model based on young reference

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.ticker import MaxNLocator

qc_path = Path(
    "/mnt/newStor/paros/paros_WORK/xintong/code/cortical_trajectory_analysis"
    "/analysis_results/figure6_new/outputs/warped_delta_382DBdHx/stage2_qc.csv"
)
qc = pd.read_csv(qc_path)

if qc.empty:
    raise ValueError(f"The QC file has no rows: {qc_path}")

# Replace these names here if the CSV uses a different abbreviation.
column_titles = {
    "n": "Count",
    "n_valid": "Valid count",
    "mean_z": "Mean z-score",
    "sd_z": "SD of z-scores",
    "z_mean": "Mean z-score",
    "z_sd": "SD of z-scores",
    "rmse": "RMSE",
    "mae": "MAE",
}

label_aliases = {
    "cn": "CN",
    "older_cn": "Older CN",
    "mci": "MCI",
    "ad": "AD",
    "outer": "Outer cortex",
    "middle": "Middle cortex",
    "inner": "Inner cortex",
    "all": "All depths",
    "with_thickness": "With thickness",
    "without_thickness": "Without thickness",
    "no_thickness": "Without thickness",
    "age_sex_thickness": "Age + sex + thickness",
    "age_sex": "Age + sex",
    "loo": "LOO",
}

def column_title(column):
    return column_titles.get(column.lower(), column.replace("_", " ").capitalize())

def format_label(value):
    if pd.isna(value):
        return "Missing"
    value = str(value)
    return label_aliases.get(value.lower(), value.replace("_", " "))

dimension_pattern = re.compile(
    r"(?:^|_)(?:model|variant|group|cohort|band|depth|split|scoring|metric)(?:_|$)",
    re.I,
)
dimension_columns = []
metrics = {}

for column in qc.columns:
    values = pd.to_numeric(qc[column], errors="coerce").replace(
        [np.inf, -np.inf], np.nan
    )
    if dimension_pattern.search(column) and (
        not values.notna().any()
        or column.lower() in {"depth", "depth_index", "depth_fraction"}
    ):
        dimension_columns.append(column)
    elif values.notna().any():
        metrics[column] = values

row_metadata = pd.DataFrame(index=qc.index)
for column in dimension_columns:
    row_metadata[column_title(column)] = qc[column].map(format_label)

if not dimension_columns or row_metadata.duplicated(keep=False).any():
    row_metadata.insert(0, "CSV row", np.arange(1, len(qc) + 1))

row_labels = row_metadata.astype(str).agg(" · ".join, axis=1)

display(Markdown("## Stage 2 quality control"))

for column, values in metrics.items():
    title = column_title(column)
    valid = values.notna()
    positions = np.arange(len(qc))

    display(Markdown(f"### {title}"))
    fig, ax = plt.subplots(
        figsize=(9, max(2.6, 0.32 * len(qc) + 1.3)),
        layout="constrained",
    )
    ax.scatter(values[valid], positions[valid], s=34, color="#275D80", zorder=3)
    ax.set_yticks(positions, row_labels)
    ax.invert_yaxis()
    ax.set_xlabel(title)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.grid(axis="x", alpha=0.25)
    plt.show()
    plt.close(fig)

    table = row_metadata.copy()
    table[title] = values
    display(table.style.format({title: "{:.6g}"}, na_rep="Missing"))

other_columns = [
    column for column in qc.columns
    if column not in dimension_columns and column not in metrics
]
for column in other_columns:
    title = column_title(column)
    display(Markdown(f"### {title}"))
    table = row_metadata.copy()
    table[title] = qc[column].map(format_label)
    display(table)
    

## 4. Export

In [ ]:
export_dir = RUN_DIR / "calibration_figures"
export_dir.mkdir(exist_ok=True)
for name, figure in {"loo_calibration": fig_calibration, "loo_depth_band_qc": fig_tails}.items():
    for ext in ("pdf", "png"):
        figure.savefig(export_dir / f"{name}.{ext}", dpi=300, bbox_inches="tight")
for name, table in {"coverage_plot": coverage_plot, "signed_z_density": density,
                    "depth_band_summary_percent": summary.reset_index(),
                    "depth_band_plot": tails}.items():
    table.to_csv(export_dir / f"{name}.csv", index=False)
print(export_dir)

## 5. Brain-age prediction: warped ΔMD versus raw MD
Shared nested 5×5 CV in 156 older CN. Each point is an out-of-fold prediction; MAE difference uses paired participant bootstrap.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'analysis_results/figure6_new').is_dir())
brain_dir = repo / 'analysis_results/figure6_new/outputs/brain_age_delta_vs_raw_md_156'
paired = pd.read_csv(brain_dir / 'paired_oof_predictions.csv')
metrics = pd.read_csv(brain_dir / 'overall_metrics.csv').set_index('representation')
boot = json.loads((brain_dir / 'paired_mae_bootstrap.json').read_text())
assert len(paired) == paired.subject_id.nunique() == 156
fig_brain_age, axes = plt.subplots(1, 2, figsize=(9, 4), sharex=True, sharey=True, constrained_layout=True)
brain_age_colors = {'delta_md': '#4B74B2', 'raw_md': '#FC8C5A'}
for ax, rep, label in zip(axes, ['delta_md', 'raw_md'], ['Warped ΔMD', 'Raw MD']):
    color = brain_age_colors[rep]
    ax.scatter(paired.actual_age, paired[f'predicted_age_{rep}'], s=22, alpha=.7, color=color)
    ax.plot([45, 85], [45, 85], '--', color='gray', linewidth=1)
    m = metrics.loc[rep]
    ax.set(title=f'{label}\nMAE={m.mae:.2f} y; RMSE={m.rmse:.2f} y; r={m.pearson_r:.2f}; R²={m.r2:.2f}', xlabel='Actual age (years)', xlim=(45,85), ylim=(45,85))
axes[0].set_ylabel('OOF predicted age (years)')
fig_brain_age.suptitle(f"156 older CN · ΔMAE (ΔMD − raw MD) = {boot['estimate']:.3f} y\nPaired bootstrap 95% CI [{boot['ci95_lower']:.3f}, {boot['ci95_upper']:.3f}]")
for ext in ['png', 'pdf']:
    fig_brain_age.savefig(brain_dir / f'brain_age_delta_vs_raw_md.{ext}', dpi=300, bbox_inches='tight')
plt.show()
